In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# ── 1. EXCHANGE LOG ─────────────────────────────────────────────────
exchange = pd.read_pickle('PERP_EXCHANGE.bz2')
print(f"Exchange log: {len(exchange)} events")
print(exchange['EventType'].value_counts())

# ── 2. MARK PRICE vs ORACLE TIME SERIES ────────────────────────────
mark_rows = exchange[exchange['EventType'] == 'MARK_PRICE_UPDATE'].copy()
mark_rows[['symbol', 'mark_price', 'oracle_price']] = mark_rows['Event'].str.split(',', expand=True)
mark_rows['mark_price'] = mark_rows['mark_price'].astype(float)
mark_rows['oracle_price'] = mark_rows['oracle_price'].astype(float)
mark_rows['premium'] = mark_rows['mark_price'] - mark_rows['oracle_price']
mark_rows['premium_bps'] = (mark_rows['premium'] / mark_rows['oracle_price']) * 10000
mark_rows = mark_rows.reset_index()  # index = EventTime

print(f"\n{'='*60}")
print(f"MARK PRICE vs ORACLE")
print(f"{'='*60}")
print(f"Updates: {len(mark_rows)}")
print(f"\nMark price:  {mark_rows['mark_price'].min():.4f} → {mark_rows['mark_price'].max():.4f}  (mean {mark_rows['mark_price'].mean():.4f})")
print(f"Oracle:      {mark_rows['oracle_price'].min():.4f} → {mark_rows['oracle_price'].max():.4f}  (mean {mark_rows['oracle_price'].mean():.4f})")
print(f"\nPremium (mark - oracle):")
print(f"  Mean:  {mark_rows['premium'].mean():+.4f}  ({mark_rows['premium_bps'].mean():+.2f} bps)")
print(f"  Std:   {mark_rows['premium'].std():.4f}  ({mark_rows['premium_bps'].std():.2f} bps)")
print(f"  Min:   {mark_rows['premium'].min():+.4f}  ({mark_rows['premium_bps'].min():+.2f} bps)")
print(f"  Max:   {mark_rows['premium'].max():+.4f}  ({mark_rows['premium_bps'].max():+.2f} bps)")

# ── 3. AGENT RESULTS ───────────────────────────────────────────────
summary = pd.read_pickle('summary_log.bz2')
print(f"\n{'='*60}")
print(f"AGENT RESULTS (from summary_log)")
print(f"{'='*60}")

if len(summary) > 0:
    # Pivot to get starting cash and ending equity per agent
    agents = {}
    for _, row in summary.iterrows():
        agent = row.get('AgentID', row.get('agent_id', None))
        etype = row.get('EventType', '')
        event = row.get('Event', '')
        if agent is not None:
            if agent not in agents:
                agents[agent] = {}
            if 'STARTING_CASH' in str(etype):
                agents[agent]['starting_cash'] = float(event)
            elif 'ENDING_EQUITY' in str(etype):
                agents[agent]['ending_equity'] = float(event)
            elif 'FINAL_BALANCE' in str(etype):
                agents[agent]['final_balance'] = float(event)

    if agents:
        df_agents = pd.DataFrame(agents).T
        if 'starting_cash' in df_agents.columns and 'ending_equity' in df_agents.columns:
            df_agents['pnl'] = df_agents['ending_equity'] - df_agents['starting_cash']
            df_agents['pnl_pct'] = df_agents['pnl'] / df_agents['starting_cash'] * 100
            print(f"\nAgents: {len(df_agents)}")
            print(f"PnL distribution:")
            print(df_agents['pnl'].describe().to_string())
            print(f"\nTop 5 winners:")
            print(df_agents.nlargest(5, 'pnl')[['starting_cash', 'ending_equity', 'pnl']].to_string())
            print(f"\nTop 5 losers:")
            print(df_agents.nsmallest(5, 'pnl')[['starting_cash', 'ending_equity', 'pnl']].to_string())
        else:
            print(summary.head(20))
    else:
        print(summary.head(20))
else:
    print("(summary_log is empty — loading per-agent logs instead)")

# ── 4. PER-AGENT LOG ANALYSIS ──────────────────────────────────────
print(f"\n{'='*60}")
print(f"PER-AGENT LOG DETAILS")
print(f"{'='*60}")

chartist_files = sorted(glob.glob('Chartist_*.bz2'))
noise_files = sorted(glob.glob('Noise_*.bz2'))

results = []
for f in chartist_files + noise_files:
    name = os.path.splitext(os.path.basename(f))[0]
    agent_type = 'Chartist' if name.startswith('Chartist') else 'Noise'
    try:
        df = pd.read_pickle(f)
        if len(df) == 0:
            continue
        starting = df[df['EventType'] == 'STARTING_CASH']['Event'].values
        equity = df[df['EventType'] == 'ENDING_EQUITY']['Event'].values
        balance = df[df['EventType'] == 'FINAL_BALANCE']['Event'].values
        results.append({
            'name': name,
            'type': agent_type,
            'events': len(df),
            'starting_cash': float(starting[0]) if len(starting) > 0 else np.nan,
            'ending_equity': float(equity[0]) if len(equity) > 0 else np.nan,
            'final_balance': float(balance[0]) if len(balance) > 0 else np.nan,
        })
    except Exception as e:
        pass

if results:
    df_res = pd.DataFrame(results)
    df_res['pnl'] = df_res['ending_equity'] - df_res['starting_cash']
    df_res['pnl_bps'] = (df_res['pnl'] / df_res['starting_cash']) * 10000

    print(f"\nLoaded {len(df_res)} agent logs")

    for t in ['Chartist', 'Noise']:
        sub = df_res[df_res['type'] == t]
        if len(sub) == 0:
            continue
        print(f"\n--- {t} agents ({len(sub)}) ---")
        print(f"  PnL mean:  {sub['pnl'].mean():+.2f}  ({sub['pnl_bps'].mean():+.2f} bps)")
        print(f"  PnL std:   {sub['pnl'].std():.2f}")
        print(f"  PnL min:   {sub['pnl'].min():+.2f}")
        print(f"  PnL max:   {sub['pnl'].max():+.2f}")
        print(f"  Winners:   {(sub['pnl'] > 0).sum()}/{len(sub)}")

    print(f"\n--- All agents ---")
    print(f"  Total PnL:     {df_res['pnl'].sum():+.2f}")
    print(f"  Mean PnL:      {df_res['pnl'].mean():+.2f}")
    print(f"  Winners:       {(df_res['pnl'] > 0).sum()}/{len(df_res)}")
    print(f"  Losers:        {(df_res['pnl'] < 0).sum()}/{len(df_res)}")

    print(f"\nTop 5 performers:")
    print(df_res.nlargest(5, 'pnl')[['name', 'type', 'ending_equity', 'pnl']].to_string(index=False))
    print(f"\nBottom 5 performers:")
    print(df_res.nsmallest(5, 'pnl')[['name', 'type', 'ending_equity', 'pnl']].to_string(index=False))
else:
    print("No agent logs found")

In [ ]:
# ── 5. ORACLE DEPLOYER LOG ──────────────────────────────────────────
deployer_log = pd.read_pickle('ORACLE_DEPLOYER.bz2')
print(f"{'='*60}")
print(f"ORACLE DEPLOYER LOG")
print(f"{'='*60}")
print(f"Rows: {len(deployer_log)}")
print(deployer_log.to_string())

# The deployer's real footprint lives in the exchange log (SET_ORACLE events)
oracle_events = exchange[exchange['EventType'] == 'SET_ORACLE'].copy()
oracle_events = oracle_events.reset_index()
oracle_events['EventTime'] = pd.to_datetime(oracle_events['EventTime'])

print(f"\n{'='*60}")
print(f"ORACLE UPDATES (SET_ORACLE in exchange log)")
print(f"{'='*60}")
print(f"Total oracle pushes:  {len(oracle_events)}")
print(f"Time range:           {oracle_events['EventTime'].min()} → {oracle_events['EventTime'].max()}")
dt = oracle_events['EventTime'].diff().dt.total_seconds().dropna()
print(f"Update interval (s):  mean={dt.mean():.2f}  std={dt.std():.4f}  min={dt.min():.2f}  max={dt.max():.2f}")

# Mark price evolution driven by oracle
mark = exchange[exchange['EventType'] == 'MARK_PRICE_UPDATE'].copy().reset_index()
mark['EventTime'] = pd.to_datetime(mark['EventTime'])
mark[['symbol', 'mark_price', 'oracle_price']] = mark['Event'].str.split(',', expand=True)
mark['mark_price'] = mark['mark_price'].astype(float)
mark['oracle_price'] = mark['oracle_price'].astype(float)

print(f"\nMark-price convergence to oracle over time:")
n_chunks = 6
chunk_size = len(mark) // n_chunks
for i in range(n_chunks):
    chunk = mark.iloc[i*chunk_size:(i+1)*chunk_size]
    spread = (chunk['mark_price'] - chunk['oracle_price']).abs()
    t0, t1 = chunk['EventTime'].iloc[0], chunk['EventTime'].iloc[-1]
    print(f"  {t0.strftime('%H:%M')}–{t1.strftime('%H:%M')}:  |mark-oracle| mean={spread.mean():.4f}  max={spread.max():.4f}")

# Funding rate summary
funding = exchange[exchange['EventType'] == 'FUNDING_SETTLED'].copy().reset_index()
funding['EventTime'] = pd.to_datetime(funding['EventTime'])
funding[['f_symbol', 'f_rate']] = funding['Event'].str.split(',', expand=True)
funding['f_rate'] = funding['f_rate'].astype(float)
print(f"\n{'='*60}")
print(f"FUNDING RATE SETTLEMENTS ({len(funding)} epochs)")
print(f"{'='*60}")
print(f"  Mean hourly rate:  {funding['f_rate'].mean():.8f}")
print(f"  Std:               {funding['f_rate'].std():.8f}")
print(f"  Min:               {funding['f_rate'].min():.8f}")
print(f"  Max:               {funding['f_rate'].max():.8f}")
print(f"\nPer-epoch rates:")
for _, r in funding.iterrows():
    print(f"  {r['EventTime'].strftime('%Y-%m-%d %H:%M')}  rate={float(r['f_rate']):+.8f}")

In [ ]:
# ── 6. SUMMARY LOG – DETAILED BREAKDOWN ─────────────────────────────
summary = pd.read_pickle('summary_log.bz2')
print(f"{'='*60}")
print(f"SUMMARY LOG  ({len(summary)} rows)")
print(f"{'='*60}")
print(f"Columns: {list(summary.columns)}")
print(f"EventTypes: {summary['EventType'].unique().tolist()}")
print(f"AgentStrategies: {summary['AgentStrategy'].unique().tolist()}")

# Pivot into one row per agent
pivoted = summary.pivot_table(index='AgentID', columns='EventType', values='Event', aggfunc='first')
pivoted.columns.name = None
for c in ['STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY']:
    if c in pivoted.columns:
        pivoted[c] = pivoted[c].astype(float)

id_to_type = {}
for _, row in summary.drop_duplicates('AgentID').iterrows():
    aid = row['AgentID']
    if 2 <= aid <= 51:
        id_to_type[aid] = 'Chartist'
    elif 52 <= aid <= 101:
        id_to_type[aid] = 'Noise'

pivoted['agent_type'] = pivoted.index.map(id_to_type)
pivoted['pnl'] = pivoted['ENDING_EQUITY'] - pivoted['STARTING_CASH']
pivoted['pnl_bps'] = (pivoted['pnl'] / pivoted['STARTING_CASH']) * 10_000
pivoted['balance_drift'] = pivoted['FINAL_BALANCE'] - pivoted['STARTING_CASH']

print(f"\n--- Per-type breakdown ---")
for t in ['Chartist', 'Noise']:
    sub = pivoted[pivoted['agent_type'] == t]
    print(f"\n  {t} ({len(sub)} agents)")
    print(f"    PnL (equity-based):  mean={sub['pnl'].mean():+.2f}  std={sub['pnl'].std():.2f}  min={sub['pnl'].min():+.2f}  max={sub['pnl'].max():+.2f}")
    print(f"    PnL (bps):           mean={sub['pnl_bps'].mean():+.2f}  std={sub['pnl_bps'].std():.2f}")
    print(f"    Balance drift:       mean={sub['balance_drift'].mean():+.2f}  (cash change excl. unrealised)")
    print(f"    Ending equity:       mean={sub['ENDING_EQUITY'].mean():.2f}  std={sub['ENDING_EQUITY'].std():.2f}")
    print(f"    Winners:             {(sub['pnl'] > 0).sum()}/{len(sub)}")

print(f"\n--- Full agent table (sorted by PnL) ---")
display_cols = ['agent_type', 'STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY', 'pnl', 'pnl_bps']
print(pivoted.sort_values('pnl', ascending=False)[display_cols].to_string())

In [ ]:
# ── 7. FUNDAMENTAL VALUE (ORACLE) TIME SERIES ───────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fund = pd.read_pickle('fundamental_ASSET-USD.bz2')
fund.index = pd.to_datetime(fund.index)

print(f"{'='*60}")
print(f"FUNDAMENTAL VALUE – ASSET-USD")
print(f"{'='*60}")
print(f"Time range:     {fund.index.min()} → {fund.index.max()}")
print(f"Duration:        {fund.index.max() - fund.index.min()}")
print(f"Data points:     {len(fund)}")
dt_f = fund.index.to_series().diff().dt.total_seconds().dropna()
print(f"Sample interval: mean={dt_f.mean():.2f}s  std={dt_f.std():.4f}s")
print(f"\nFundamental value statistics:")
print(fund['FundamentalValue'].describe().to_string())
print(f"\nFirst/last values: {fund['FundamentalValue'].iloc[0]:.4f} → {fund['FundamentalValue'].iloc[-1]:.4f}")
print(f"Total change:      {fund['FundamentalValue'].iloc[-1] - fund['FundamentalValue'].iloc[0]:+.4f}")

# Reuse mark price data from cell 0
mark = exchange[exchange['EventType'] == 'MARK_PRICE_UPDATE'].copy().reset_index()
mark['EventTime'] = pd.to_datetime(mark['EventTime'])
mark[['symbol', 'mark_price', 'oracle_price']] = mark['Event'].str.split(',', expand=True)
mark['mark_price'] = mark['mark_price'].astype(float)
mark['oracle_price'] = mark['oracle_price'].astype(float)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1, 1]})

# Panel 1: Fundamental, Oracle (as seen by exchange), and Mark Price
ax1 = axes[0]
ax1.plot(fund.index, fund['FundamentalValue'], label='Fundamental (CSV oracle)', linewidth=1, alpha=0.8)
ax1.plot(mark['EventTime'], mark['oracle_price'], label='Oracle (exchange view)', linewidth=0.8, alpha=0.7, linestyle='--')
ax1.plot(mark['EventTime'], mark['mark_price'], label='Mark price', linewidth=0.8, alpha=0.7)
ax1.set_ylabel('Price')
ax1.set_title('Fundamental Value, Oracle Price & Mark Price')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)

# Panel 2: Premium (mark - oracle) in bps
premium_bps = (mark['mark_price'] - mark['oracle_price']) / mark['oracle_price'] * 10_000
ax2 = axes[1]
ax2.fill_between(mark['EventTime'], premium_bps, 0, alpha=0.4, color='C3')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('Premium (bps)')
ax2.set_title('Mark-Oracle Premium')
ax2.grid(True, alpha=0.3)

# Panel 3: Fundamental returns
fund_ret = fund['FundamentalValue'].pct_change().dropna() * 10_000
ax3 = axes[2]
ax3.plot(fund_ret.index, fund_ret, linewidth=0.5, alpha=0.7, color='C2')
ax3.axhline(0, color='black', linewidth=0.5)
ax3.set_ylabel('Return (bps)')
ax3.set_title('Fundamental Value – Period Returns')
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

In [ ]:
# ── 8. ORDER BOOK ACTIVITY – ONE RANDOM AGENT PER TYPE ──────────────
# Note: log_orders was disabled for this run, so per-agent order logs
# are not available. We use QUERY_MARK_PRICE events from the exchange
# log to reconstruct each agent's activity timeline (every wakeup that
# queried the mark price), overlaid on the mark price time series.

import random
random.seed(42)

# Agent ID mapping: 2-51 = Chartist_0..49, 52-101 = Noise_0..49
chartist_ids = list(range(2, 52))
noise_ids = list(range(52, 102))

pick_chartist = random.choice(chartist_ids)
pick_noise = random.choice(noise_ids)
chartist_name = f"Chartist_{pick_chartist - 2}"
noise_name = f"Noise_{pick_noise - 52}"

queries = exchange[exchange['EventType'] == 'QUERY_MARK_PRICE'].copy().reset_index()
queries['EventTime'] = pd.to_datetime(queries['EventTime'])
queries['AgentID'] = queries['Event'].astype(int)

# Mark price at each query time (nearest oracle update)
mark = exchange[exchange['EventType'] == 'MARK_PRICE_UPDATE'].copy().reset_index()
mark['EventTime'] = pd.to_datetime(mark['EventTime'])
mark[['symbol', 'mark_price', 'oracle_price']] = mark['Event'].str.split(',', expand=True)
mark['mark_price'] = mark['mark_price'].astype(float)
mark['oracle_price'] = mark['oracle_price'].astype(float)

# Per-agent PnL from summary
pivoted_pnl = summary.pivot_table(index='AgentID', columns='EventType', values='Event', aggfunc='first')
pivoted_pnl.columns.name = None
for c in ['STARTING_CASH', 'ENDING_EQUITY']:
    pivoted_pnl[c] = pivoted_pnl[c].astype(float)
pivoted_pnl['pnl'] = pivoted_pnl['ENDING_EQUITY'] - pivoted_pnl['STARTING_CASH']

agents_to_plot = [
    (pick_chartist, chartist_name, 'Chartist', 'C0'),
    (pick_noise, noise_name, 'Noise', 'C1'),
]

fig, axes = plt.subplots(len(agents_to_plot), 1, figsize=(14, 5 * len(agents_to_plot)), sharex=True)
if len(agents_to_plot) == 1:
    axes = [axes]

for ax, (aid, aname, atype, color) in zip(axes, agents_to_plot):
    agent_queries = queries[queries['AgentID'] == aid]
    pnl_val = pivoted_pnl.loc[aid, 'pnl'] if aid in pivoted_pnl.index else float('nan')
    n_queries = len(agent_queries)

    # Compute inter-wakeup intervals
    wake_dt = agent_queries['EventTime'].diff().dt.total_seconds().dropna()

    ax.plot(mark['EventTime'], mark['mark_price'], color='grey', alpha=0.4,
            linewidth=0.8, label='Mark price')
    ax.plot(mark['EventTime'], mark['oracle_price'], color='black', alpha=0.3,
            linewidth=0.8, linestyle='--', label='Oracle price')

    # Show each wakeup as a vertical tick
    ax.vlines(agent_queries['EventTime'], ymin=mark['mark_price'].min() * 0.998,
              ymax=mark['mark_price'].min() * 0.998 + (mark['mark_price'].max() - mark['mark_price'].min()) * 0.05,
              colors=color, alpha=0.6, linewidth=0.5, label=f'Wakeups ({n_queries})')

    ax.set_ylabel('Price')
    ax.set_title(
        f'{aname} (ID {aid}, {atype})  –  '
        f'{n_queries} wakeups, '
        f'interval ~{wake_dt.mean():.1f}s ± {wake_dt.std():.1f}s, '
        f'PnL = {pnl_val:+.2f}'
    )
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Print summary table for the selected agents
print(f"\n{'='*60}")
print(f"SELECTED AGENT DETAILS")
print(f"{'='*60}")
for aid, aname, atype, _ in agents_to_plot:
    agent_queries = queries[queries['AgentID'] == aid]
    wake_dt = agent_queries['EventTime'].diff().dt.total_seconds().dropna()
    pnl_val = pivoted_pnl.loc[aid, 'pnl'] if aid in pivoted_pnl.index else float('nan')
    eq_val = pivoted_pnl.loc[aid, 'ENDING_EQUITY'] if aid in pivoted_pnl.index else float('nan')

    print(f"\n  {aname} (Agent ID {aid}, type={atype})")
    print(f"    Total wakeups:       {len(agent_queries)}")
    print(f"    First wakeup:        {agent_queries['EventTime'].min()}")
    print(f"    Last wakeup:         {agent_queries['EventTime'].max()}")
    print(f"    Mean interval:       {wake_dt.mean():.2f}s")
    print(f"    Ending equity:       {eq_val:,.2f}")
    print(f"    PnL:                 {pnl_val:+.2f}")